# Colab 09 — Failed Auction V2 (FROZEN ONE-CANDIDATE RESEARCH)

**OFFLINE only:** owner-private pre-existing MICRO ZIP; never mount Google Drive here. No original CryptoResearchLab source reads/writes, no 2025 Validation or 2026 OOS. Reused July = exploratory; August = separately owner-approved, one-shot Development test, never proof of live profitability. Save all negative and UNKNOWN attempts privately.

- One fixed rule. No optimizing stop/thresholds after July results.
- Entry/exit are first-trade price proxies **not executable quotes/fills**.
- Save plan before calculating outcomes, save freeze token and exact public code commit.


In [ ]:
# 1. New offline runtime. Public CODE clone only. No Drive credentials.
import os, sys, subprocess, json, hashlib
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('Drive mounted: disconnect/delete runtime; use no-Drive Colab')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Credentials detected; fail closed')
repo=Path('/content/hypolab-failed-auction-v2-public-code')
if repo.exists():raise RuntimeError('Use fresh Colab runtime; do not reuse old code checkout')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
sys.path.insert(0,str(repo/'src'))
from hypolab.strategy_v2 import make_strategy_v2_plan,run_strategy_v2,freeze_token_sha256,DIAGNOSTIC_MONTH,ONE_SHOT_DEVELOPMENT_MONTH
from hypolab.micro_bundle import MAX_ARCHIVE_BYTES
from google.colab import files
print('Public CODE commit:',commit)
print('NO Drive mounted. Historical C1/C2 source inaccessible to this notebook.')


In [ ]:
# 2. Verify ONE existing owner-private MICRO ZIP BEFORE any research plan.
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Exactly one MICRO ZIP required')
name,raw=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_MICRO_DEV_') or not name.endswith('.zip') or not 0<len(raw)<=MAX_ARCHIVE_BYTES:
    raise RuntimeError('Unexpected MICRO ZIP name or size')
expected=input('Exact MICRO ZIP SHA-256: ').strip().lower()
actual=hashlib.sha256(raw).hexdigest()
if expected!=actual:raise RuntimeError('ZIP SHA-256 MISMATCH: no analysis')
private=Path('/content/hypolab-failed-auction-v2-private')
private.mkdir(mode=0o700,exist_ok=False)
packet=private/'verified-input.zip'
with packet.open('xb') as f:
    os.chmod(packet,0o600)
    f.write(raw)
del uploaded,raw
plan=make_strategy_v2_plan(packet)
p=json.loads(plan)
print('Verified DEVELOPMENT month:',p['month'],'kind:',p['role'])
print('FROZEN STRATEGY V2 TOKEN SHA256:',p['freeze_token_sha256'])
if p['month']==ONE_SHOT_DEVELOPMENT_MONTH:
    july_token=input('Paste saved JULY frozen STRATEGY V2 token (not V3.1 token): ').strip().lower()
    if july_token!=p['freeze_token_sha256']:
        raise RuntimeError('Strategy/code drift since July: August one-shot BLOCKED')
elif p['month']!=DIAGNOSTIC_MONTH:raise RuntimeError('Month not permitted')


In [ ]:
# 3. Download frozen plan BEFORE opening trading outcomes.
plan_file=private/'strategy-v2-frozen-plan.json'
with plan_file.open('xb') as f:f.write(plan)
print('Plan SHA-256:',hashlib.sha256(plan).hexdigest())
print('SAVE FROZEN STRATEGY V2 TOKEN:',p['freeze_token_sha256'])
files.download(str(plan_file))


In [ ]:
# 4. ONE fixed strategy with ALL negative/UNKNOWN outcomes.
report=run_strategy_v2(packet,plan)
print('RESULT',report['month'],report['model_status'])
print('Raw signals:',report['raw_signals'],'Completed trades:',report['primary']['trades'])
print('Mean gross bps:',report['primary']['mean_gross_bps'])
print('Mean net bps:',report['primary']['mean_net_bps'])
print('Day-cluster 95pct descriptive range:',report['primary']['daily_cluster_95pct_net_mean'])
print('Stress mean net bps:',report['stress_mean_net_bps'])
print('Opposite control:',report['opposite_same_case_control']['mean_net_bps'])
print('Skips / UNKNOWN:',report['skips'])
print('PREDECLARED GATES:')
for key,value in report['gates'].items():print(' ',key,value)
print('Theoretical model only. Executable profitability is NOT established.')


In [ ]:
# 5. Descriptive cumulative per-unit P&L — no cherry-picked winners.
import matplotlib.pyplot as plt
def accumulated(trades):
    total=0.;out=[0.]
    for tr in trades:
        total+=tr['net_bps'];out.append(total)
    return out
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(accumulated(report['trades']),label='V2 primary (assumed costs)')
ax.plot(accumulated(report['opposite_side_control_trades']),label='Opposite-side same-entry control (NOT independent schedule)')
ax.set(xlabel='Trade sequence (NOT calendar time)',ylabel='Cumulative unit bps',
       title='DESCRIPTIVE MODEL ONLY — synthetic executed-trade price proxies, not real P&L')
ax.grid(True,alpha=.2);ax.legend();plt.show()
if not report['trades']:print('NO COMPLETED TRADES: valid result; DO NOT tune July thresholds')


In [ ]:
# 6. Download private complete report, idempotent and NEVER overwrite.
content=json.dumps(report,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n'
report_file=private/'strategy-v2-full-private-report.json'
if report_file.exists() and report_file.read_text(encoding='utf8')!=content:
    h=hashlib.sha256(content.encode('utf8')).hexdigest()[:12]
    report_file=private/f'strategy-v2-full-private-report-{h}.json'
if not report_file.exists():
    with report_file.open('x',encoding='utf8') as f:f.write(content)
if report_file.read_text(encoding='utf8')!=content:raise RuntimeError('REPORT HASH CONTENT MISMATCH')
print('COMPLETE PRIVATE REPORT:',report_file.name)
files.download(str(report_file))
print('Store ZIP, July token, plan, report PRIVATELY. Never commit data/results to GitHub.')
